# 03. Reversal Patterns

Reversing pointers is the most asked linked list skill. Master the 3-pointer loop, and all variations follow.

**Learning order:** REVERSE → RECURSIVE → BETWEEN M AND N → K-GROUPS → ROTATE → REORDER → INTERVIEW THINKING

## 1. Reverse a list (iterative)

Three variables: `prev`, `cur`, `nxt`.
1. Save `nxt = cur.next` (or you lose the rest).
2. Flip: `cur.next = prev`.
3. Move: `prev = cur`, `cur = nxt`.

At the end `prev` is the new head. Real life: turning around a train so the last coach becomes the engine.

```
None <- 1 <- 2    3 -> 4        prev=2, cur=3
```

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(vals):                      # list -> linked list
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def to_list(head):                    # linked list -> list (for printing)
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

def reverse_list(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    return prev

print(to_list(reverse_list(build([1, 2, 3, 4, 5]))))   # [5, 4, 3, 2, 1]

[5, 4, 3, 2, 1]


## 2. Recursive version

Trust the recursion: `reverse(head.next)` returns the reversed rest, with `head.next` now being its **tail**. Make that tail point back to `head`, and cut `head`'s own pointer.

Time O(n), space O(n) for the call stack.

In [2]:
def reverse_recursive(head):
    if not head or not head.next:
        return head
    new_head = reverse_recursive(head.next)
    head.next.next = head
    head.next = None
    return new_head

print(to_list(reverse_recursive(build([1, 2, 3]))))   # [3, 2, 1]

[3, 2, 1]


## 3. Reverse between positions `left` and `right`

1. Walk `prev` to the node just before `left` (use a dummy node).
2. Repeat `right - left` times: take the node after `cur` and **move it to the front** of the reversed part.

In [3]:
def reverse_between(head, left, right):
    dummy = ListNode(0, head)
    prev = dummy
    for _ in range(left - 1):
        prev = prev.next
    cur = prev.next
    for _ in range(right - left):
        nxt = cur.next
        cur.next = nxt.next
        nxt.next = prev.next
        prev.next = nxt
    return dummy.next

print(to_list(reverse_between(build([1, 2, 3, 4, 5]), 2, 4)))   # [1, 4, 3, 2, 5]

[1, 4, 3, 2, 5]


## 4. Reverse in K-groups

Reverse every block of `k` nodes. If fewer than `k` remain, leave them as they are.
For each block: check there are `k` nodes (`get_kth`), reverse the block, reconnect with the previous block and the next block.

In [4]:
def get_kth(cur, k):
    while cur and k > 0:
        cur = cur.next
        k -= 1
    return cur

def reverse_k_group(head, k):
    dummy = ListNode(0, head)
    group_prev = dummy
    while True:
        kth = get_kth(group_prev, k)
        if not kth:
            break
        group_next = kth.next
        prev, cur = group_next, group_prev.next       # prev starts as the node after the block
        while cur is not group_next:
            nxt = cur.next
            cur.next = prev
            prev = cur
            cur = nxt
        tmp = group_prev.next                         # old first node is now the block's tail
        group_prev.next = kth
        group_prev = tmp
    return dummy.next

print(to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 2)))   # [2, 1, 4, 3, 5]
print(to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 3)))   # [3, 2, 1, 4, 5]

[2, 1, 4, 3, 5]
[3, 2, 1, 4, 5]


## 5. Rotate List

Rotate right by `k`. Make the list circular, then break it at the right place: new tail is at index `n - k % n - 1`.

In [5]:
def rotate_right(head, k):
    if not head or not head.next:
        return head
    n, tail = 1, head
    while tail.next:
        tail = tail.next
        n += 1
    k %= n
    if k == 0:
        return head
    tail.next = head                                  # make circular
    new_tail = head
    for _ in range(n - k - 1):
        new_tail = new_tail.next
    new_head = new_tail.next
    new_tail.next = None
    return new_head

print(to_list(rotate_right(build([1, 2, 3, 4, 5]), 2)))   # [4, 5, 1, 2, 3]

[4, 5, 1, 2, 3]


## 6. Reorder List: `L0 → Ln → L1 → Ln-1 → ...`

Combines three skills: **find middle**, **reverse the second half**, **merge alternately**. In place, O(1) space.

In [6]:
def reorder_list(head):
    if not head or not head.next:
        return
    slow = fast = head
    while fast.next and fast.next.next:
        slow = slow.next
        fast = fast.next.next
    second = reverse_list(slow.next)
    slow.next = None                                  # cut the list in two
    first = head
    while second:
        t1, t2 = first.next, second.next
        first.next = second
        second.next = t1
        first, second = t1, t2

h = build([1, 2, 3, 4, 5]); reorder_list(h)
print(to_list(h))   # [1, 5, 2, 4, 3]
h = build([1, 2, 3, 4]); reorder_list(h)
print(to_list(h))   # [1, 4, 2, 3]

[1, 5, 2, 4, 3]
[1, 4, 2, 3]


## 7. Interview thinking

**Signals:** "reverse", "k at a time", "between positions", "rotate", "palindrome", "reorder".

**Say:** "I'll reverse with three pointers, saving `next` before flipping. O(n) time, O(1) space."

**Common mistakes:** overwriting `cur.next` before saving it, returning `head` instead of `prev`, forgetting to cut the first half (`slow.next = None`) → cycle, wrong gap in `reverse_between`, not using a dummy node when `left == 1`.

## Quick Revision

- Reverse: `nxt = cur.next; cur.next = prev; prev = cur; cur = nxt`. Return `prev`.
- Recursive: `head.next.next = head; head.next = None`.
- Between: dummy, walk to `left-1`, move next node to front `right-left` times.
- K-group: check k nodes exist, reverse, reconnect.
- Reorder = middle + reverse + merge.